# 02 — Next-Year Forecasting Data Preparation
**Project:** Crop Yield Prediction and Forecasting Using Agricultural Data and Machine Learning  
*Author: Aryan Puranik*

This notebook prepares the historical agricultural panel for **next-year crop yield forecasting**:
$$\text{Forecast } \text{Yield}_{r, c, t+1} \text{ at forecast origin } t \text{ using only information available up to year } t.$$

### Research Protocol & Non-Negotiable Rules
1. **Forecast Horizon Alignment:** We construct exact next-calendar-year targets ($t+1$) rather than simple row shifts across missing survey years.
2. **Chronological Splitting by Target Year:**
   - **Training Partition:** $\text{Target\_Year} \le 2011$ (Prediction origin years $1966 \le t \le 2010$)
   - **Validation Partition:** $\text{Target\_Year} \in [2012, 2014]$ (Prediction origin years $2011 \le t \le 2013$)
   - **Test Partition:** $\text{Target\_Year} \in [2015, 2017]$ (Prediction origin years $2014 \le t \le 2016$, reserved and untouched)
3. **No Predictive ML Model Fitting:** In strict accordance with research guidelines, no predictive machine learning models are trained in this notebook.
4. **Leakage-Safe Preprocessing:** All imputers, encoders, and scalers are fitted strictly on the training partition.


## 1. Setup, Imports, and Data Loading


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "crop_yield_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_PATH.is_file(), f"Dataset not found: {RAW_PATH}"

df = pd.read_csv(RAW_PATH)
print(f"Raw Dataset Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Calendar Year Span: {df['Year'].min()} - {df['Year'].max()}")
df.head(3)


Raw Dataset Loaded: 198,369 rows x 15 columns
Calendar Year Span: 1966 - 2017


## 2. Panel Key and Missing-Year Gap Audit

In agricultural panel data, successive survey rows for a given district and crop do not always represent contiguous calendar years. If a survey was discontinued or skipped for 2 or more years, a naive row shift (`df.groupby().shift(-1)`) would assign the yield of year $t+2$ or $t+5$ as the next-year target, creating severe target contamination.

We audit the temporal intervals between successive observations per `(Region, Crop)` panel.


In [4]:
KEY = ["Region", "Crop", "Year"]
assert not df.duplicated(KEY).any(), "Duplicate keys detected in raw data!"

# Sort panel strictly by district, crop, and year
panel = df.sort_values(KEY).copy()
panel["Year_Next_Row"] = panel.groupby(["Region", "Crop"])["Year"].shift(-1)
panel["Year_Gap"] = panel["Year_Next_Row"] - panel["Year"]

gap_summary = panel["Year_Gap"].value_counts().sort_index().head(6)
print("Intervals between successive observations in the same district-crop panel:")
display(pd.DataFrame({"Count": gap_summary, "Percentage (%)": (gap_summary / len(panel) * 100).round(2)}))

corrupted_rows = int((panel["Year_Gap"] > 1).sum())
print(f"\nMethodological Warning: A naive shift(-1) would create invalid multi-year targets for {corrupted_rows:,} rows.")
print("We strictly reject row shifting in favor of exact calendar year joins.")


Intervals between successive observations in the same district-crop panel:
           Count  Percentage (%)
Year_Gap                        
1.0       185685           93.61
2.0         3481            1.75
3.0         1340            0.68
4.0          647            0.33
5.0          286            0.14
6.0          403            0.20

Methodological Warning: A naive shift(-1) would create invalid multi-year targets for 7,355 rows.
We strictly reject row shifting in favor of exact calendar year joins.


## 3. Exact Next-Calendar-Year Target Construction

We create the explicit forecasting relationship:
$$\text{Target\_Year} = \text{Year} + 1$$
We then merge the exact observed yield at year $t+1$ as `Target_Yield`.
- For observation at year $t$, `Yield` represents the currently published yield (available at origin $t$).
- `Target_Yield` represents the true yield realized at year $t+1$ (the forecasting target).
- Rows where year $t+1$ was not surveyed or where $t=2017$ correctly receive `NaN` for `Target_Yield`.


In [6]:
# Build lookup table for next-calendar-year yield
next_year_lookup = df[["Region", "Crop", "Year", "Yield"]].copy()
next_year_lookup = next_year_lookup.rename(columns={
    "Year": "Target_Year",
    "Yield": "Target_Yield"
})

# Align forecast origin year t to target year t + 1
forecast_panel = df.copy()
forecast_panel["Target_Year"] = forecast_panel["Year"] + 1

forecast_panel = forecast_panel.merge(
    next_year_lookup,
    on=["Region", "Crop", "Target_Year"],
    how="left",
    validate="many_to_one"
)

print(f"Total panel rows: {len(forecast_panel):,}")
print(f"Rows with verified next-year target available: {forecast_panel['Target_Yield'].notna().sum():,} ({forecast_panel['Target_Yield'].notna().mean()*100:.1f}%)")
print(f"Rows with unobserved next-year target (gaps or horizon t=2017): {forecast_panel['Target_Yield'].isna().sum():,}")

# Display verification of exact alignment
display(forecast_panel[["Region", "Crop", "Year", "Yield", "Target_Year", "Target_Yield"]].head(6))


Total panel rows: 198,369
Rows with verified next-year target available: 185,685 (93.6%)
Rows with unobserved next-year target (gaps or horizon t=2017): 12,684
                     Region           Crop  ...  Target_Year  Target_Yield
0  Anantapur, AndhraPradesh         Castor  ...         1967        195.40
1  Anantapur, AndhraPradesh       Chickpea  ...         1967        333.33
2  Anantapur, AndhraPradesh         Cotton  ...         1967         28.57
3  Anantapur, AndhraPradesh  Finger Millet  ...         1967        972.22
4  Anantapur, AndhraPradesh      Groundnut  ...         1967        635.56
5  Anantapur, AndhraPradesh          Maize  ...         1967       1000.00

[6 rows x 6 columns]


## 4. Chronological Partitioning by Target_Year

In next-year forecasting, partitioning must be defined by **`Target_Year`**, not by the origin year:
- **Training Set:** $\text{Target\_Year} \le 2011$ (Origin years $1966 \le t \le 2010$)
- **Validation Set:** $\text{Target\_Year} \in [2012, 2014]$ (Origin years $2011 \le t \le 2013$)
- **Test Set:** $\text{Target\_Year} \in [2015, 2017]$ (Origin years $2014 \le t \le 2016$)

This guarantees zero temporal leakage: no record predicting 2012 can be seen during training.


In [8]:
# Masks for evaluated records (having observed target)
valid_target = forecast_panel["Target_Yield"].notna()

train_mask = (forecast_panel["Target_Year"] <= 2011) & valid_target
val_mask = forecast_panel["Target_Year"].between(2012, 2014) & valid_target
test_mask = forecast_panel["Target_Year"].between(2015, 2017) & valid_target

train_panel = forecast_panel[train_mask].copy()
val_panel = forecast_panel[val_mask].copy()
test_panel = forecast_panel[test_mask].copy()

# Integrity checks
assert len(train_panel) + len(val_panel) + len(test_panel) == valid_target.sum()
assert train_panel["Target_Year"].max() <= 2011
assert val_panel["Target_Year"].min() == 2012 and val_panel["Target_Year"].max() == 2014
assert test_panel["Target_Year"].min() == 2015 and test_panel["Target_Year"].max() == 2017
assert train_panel["Target_Year"].max() < val_panel["Target_Year"].min()
assert val_panel["Target_Year"].max() < test_panel["Target_Year"].min()

print("Partition Dimensions:")
print(f"  Training:   {len(train_panel):,} records | Origin Years: {train_panel['Year'].min()}-{train_panel['Year'].max()} -> Target Years: {train_panel['Target_Year'].min()}-{train_panel['Target_Year'].max()}")
print(f"  Validation: {len(val_panel):,} records  | Origin Years: {val_panel['Year'].min()}-{val_panel['Year'].max()} -> Target Years: {val_panel['Target_Year'].min()}-{val_panel['Target_Year'].max()}")
print(f"  Test:       {len(test_panel):,} records  | Origin Years: {test_panel['Year'].min()}-{test_panel['Year'].max()} -> Target Years: {test_panel['Target_Year'].min()}-{test_panel['Target_Year'].max()}")

# Timeline Visualization
fig, ax = plt.subplots(figsize=(10, 2.8))
for start, end, label, color in [(1967, 2011, "Training Targets (<= 2011)", "#1f77b4"),
                                  (2012, 2014, "Validation Targets (2012-2014)", "#ff7f0e"),
                                  (2015, 2017, "Test Targets (2015-2017)", "#2ca02c")]:
    ax.barh("Target Timeline", end - start + 1, left=start - 0.5, label=label, color=color, alpha=0.85)

ax.set_xlim(1965, 2018)
ax.set_xlabel("Target Calendar Year (t + 1)")
ax.set_title("Chronological Partitioning by Forecast Target Year")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=3)
plt.tight_layout()
plt.show()


Partition Dimensions:
  Training:   164,915 records | Origin Years: 1966-2010 -> Target Years: 1967-2011
  Validation: 10,610 records  | Origin Years: 2011-2013 -> Target Years: 2012-2014
  Test:       10,160 records  | Origin Years: 2014-2016 -> Target Years: 2015-2017


## 5. Crop and Regional Coverage Across Partitions

We verify that crop representation remains balanced across the training, validation, and test partitions.


In [10]:
coverage_table = pd.DataFrame({
    "Training_Records": train_panel.groupby("Crop")["Target_Yield"].count(),
    "Validation_Records": val_panel.groupby("Crop")["Target_Yield"].count(),
    "Test_Records": test_panel.groupby("Crop")["Target_Yield"].count(),
    "Training_Districts": train_panel.groupby("Crop")["Region"].nunique(),
    "Validation_Districts": val_panel.groupby("Crop")["Region"].nunique()
}).fillna(0).astype(int).sort_values("Training_Records", ascending=False)

print("Crop Coverage Across Partitions:")
display(coverage_table)


Crop Coverage Across Partitions:
                    Training_Records  ...  Validation_Districts
Crop                                  ...                      
Rice                           12431  ...                   271
Maize                          11489  ...                   265
Chickpea                       11342  ...                   248
Sugarcane                      11324  ...                   245
Wheat                          11304  ...                   242
Minor Pulses                   10989  ...                   270
Sesamum                        10714  ...                   251
Pigeonpea                      10669  ...                   247
Groundnut                       9949  ...                   222
Sorghum (Kharif)                9395  ...                   180
Rapeseed & Mustard              9178  ...                   217
Pearl Millet                    8260  ...                   172
Barley                          6473  ...                   146
Cotton 

## 6. Outlier Policy for Target_Yield (Documented, No Automatic Deletion)

In Phase 01, two Chickpea 2007 entries (> 10,000 kg/ha) were identified as probable data-entry typos:
- `Muzaffarpur, Bihar`: 58,363.64 kg/ha
- `Saharsa, Bihar`: 32,000.00 kg/ha

In next-year forecasting, these anomalies appear when origin year is $t = 2006$ (predicting Target_Year $t+1 = 2007$).
**Documented Policy:**
1. We flag these two candidate anomalies in the training set and record their keys.
2. In the uncleaned training baseline, all rows are retained.
3. In subsequent model training experiments, we test the impact of masking these two verified typos in training error loss.
4. Validation and test targets are **never modified or removed**.


In [12]:
target_anomalies = train_panel[
    (train_panel["Crop"] == "Chickpea") &
    (train_panel["Target_Year"] == 2007) &
    (train_panel["Target_Yield"] > 10000)
]

print(f"Identified {len(target_anomalies)} extreme Chickpea Target_Yield anomalies in training partition:")
display(target_anomalies[["Region", "Crop", "Year", "Yield", "Target_Year", "Target_Yield"]])

# Add boolean flag column to keep data quality visible
for p in [train_panel, val_panel, test_panel]:
    p["Is_Suspected_Target_Anomaly"] = (
        (p["Crop"] == "Chickpea") &
        (p["Target_Year"] == 2007) &
        (p["Target_Yield"] > 10000)
    ).astype(int)

print(f"Anomaly flag recorded across all partitions. Total flagged rows in training: {train_panel['Is_Suspected_Target_Anomaly'].sum()}")


Identified 1 extreme Chickpea Target_Yield anomalies in training partition:
                   Region      Crop  Year   Yield  Target_Year  Target_Yield
15756  Muzaffarpur, Bihar  Chickpea  2006  844.54         2007      58363.64
Anomaly flag recorded across all partitions. Total flagged rows in training: 1


## 7. Leakage-Safe Origin Feature Preprocessing (Training-Only Fit)

We establish the foundational preprocessing for origin-year features ($X_t$):
- **Numeric Features:** Imputed with median and scaled using `StandardScaler`.
- **Categorical Features:** Imputed with most frequent and encoded using `OneHotEncoder(handle_unknown='ignore')`.

All transformation parameters (means, standard deviations, categorical vocabularies) are learned **strictly on the training partition** ($t+1 \le 2011$).


In [14]:
origin_numeric_cols = [
    "Year", "Temperature", "Rainfall", "Humidity", "pH",
    "Nitrogen", "Phosphorus", "Potassium", "Pesticide", "Yield"
]
origin_categorical_cols = ["Region", "Crop", "Soil Type"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), origin_numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), origin_categorical_cols)
])

# Fit strictly on training panel
preprocessor.fit(train_panel[origin_numeric_cols + origin_categorical_cols])

# Transform training and validation sets
X_train_proc = preprocessor.transform(train_panel[origin_numeric_cols + origin_categorical_cols])
X_val_proc = preprocessor.transform(val_panel[origin_numeric_cols + origin_categorical_cols])

feature_names = preprocessor.get_feature_names_out()
print("Preprocessing Verification:")
print(f"  Training matrix shape:   {X_train_proc.shape}")
print(f"  Validation matrix shape: {X_val_proc.shape}")
print(f"  Total encoded features:  {len(feature_names)}")
print(f"  Missing values check:    Training NaNs = {np.isnan(X_train_proc).sum()}, Val NaNs = {np.isnan(X_val_proc).sum()}")
print(f"  Infinite values check:   Training Infs = {np.isinf(X_train_proc).sum()}, Val Infs = {np.isinf(X_val_proc).sum()}")


Preprocessing Verification:
  Training matrix shape:   (164915, 346)
  Validation matrix shape: (10610, 346)
  Total encoded features:  346
  Missing values check:    Training NaNs = 0, Val NaNs = 0
  Infinite values check:   Training Infs = 0, Val Infs = 0


## 8. Export Prepared Forecast Partitions

We export the next-year forecasting panels to `data/processed/` for hypothesis-driven feature engineering in Phase 03:
- `train_forecast_panel.csv` ($N=164,915$)
- `val_forecast_panel.csv` ($N=10,610$)
- `test_forecast_panel.csv` ($N=10,160$)
- `forecast_target_metadata.json`


In [16]:
# Save processed CSVs
train_panel.to_csv(PROCESSED_DIR / "train_forecast_panel.csv", index=False)
val_panel.to_csv(PROCESSED_DIR / "val_forecast_panel.csv", index=False)
test_panel.to_csv(PROCESSED_DIR / "test_forecast_panel.csv", index=False)

metadata = {
    "problem_formulation": "Next-year crop yield forecasting (t+1)",
    "target_variable": "Target_Yield",
    "target_year_variable": "Target_Year",
    "origin_year_variable": "Year",
    "partitions": {
        "training": {
            "target_years": "1967-2011",
            "origin_years": "1966-2010",
            "record_count": len(train_panel)
        },
        "validation": {
            "target_years": "2012-2014",
            "origin_years": "2011-2013",
            "record_count": len(val_panel)
        },
        "test": {
            "target_years": "2015-2017",
            "origin_years": "2014-2016",
            "record_count": len(test_panel)
        }
    },
    "provenance": "Exact calendar year joins (Year + 1) avoiding 7,355 multi-year gap errors"
}

with open(PROCESSED_DIR / "forecast_target_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved prepared panels and metadata successfully to data/processed/")


Saved prepared panels and metadata successfully to data/processed/


## 9. Research Summary and Handoff

1. **Exact-Calendar Forecast Targets:** Next-year yield ($t+1$) was matched by exact calendar years, preventing 7,355 multi-year gap corruption errors that simple row shifts would have introduced.
2. **Disjoint Chronological Partitions:** Splitting by `Target_Year` establishes strict temporal separation:
   - Training: Target_Year $\le 2011$ ($N=164,915$)
   - Validation: Target_Year $2012-2014$ ($N=10,610$)
   - Test: Target_Year $2015-2017$ ($N=10,160$)
3. **Zero Predictive ML Fitting:** No predictive regression models were fitted, maintaining complete research hygiene.
4. **Handoff to Phase 03:** These prepared forecast panels provide the verified foundation for hypothesis-driven feature engineering (historical lag momentum, within-district climate anomalies, and regional baselines).
